# 01 - Data Exploration
Download BBBC021 images and metadata, then inspect metadata and create Group A vs Group B labels.

In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

RAW_IMAGES_DIR = PROJECT_ROOT / 'data' / 'raw' / 'images'
RAW_META_DIR = PROJECT_ROOT / 'data' / 'raw' / 'metadata'
RAW_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
RAW_META_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_ZIP_URLS = [
    'https://data.broadinstitute.org/bbbc/BBBC021/BBBC021_v1_images_Week1_22123.zip',
]
CSV_URL = 'https://data.broadinstitute.org/bbbc/BBBC021/BBBC021_v1_image.csv'
IMAGES_ZIP_DIR = PROJECT_ROOT / 'data' / 'raw'
IMAGES_ZIP_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RAW_META_DIR / 'BBBC021_v1_image.csv'

print('Project root:', PROJECT_ROOT)

In [ ]:
for zip_url in IMAGE_ZIP_URLS:
    zip_name = Path(zip_url).name
    zip_path = IMAGES_ZIP_DIR / zip_name
    if not zip_path.exists():
        urllib.request.urlretrieve(zip_url, zip_path)

if not any(RAW_IMAGES_DIR.rglob('*.tif')):
    for zip_url in IMAGE_ZIP_URLS:
        zip_name = Path(zip_url).name
        zip_path = IMAGES_ZIP_DIR / zip_name
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(RAW_IMAGES_DIR)

if not CSV_PATH.exists():
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)

print('Images:', len(list(RAW_IMAGES_DIR.rglob('*.tif'))))
print('Metadata CSV exists:', CSV_PATH.exists())

In [ ]:
from data_loader import load_metadata, assign_group_labels

meta = load_metadata(CSV_PATH)
meta = assign_group_labels(meta)

display(meta.head())
print('Rows:', len(meta))
print('Columns:', len(meta.columns))
print(meta['group'].value_counts(dropna=False))